In [43]:
import pandas as pd
import re

df = pd.read_excel("DeepX_train.xlsx")
print(df.head())

   review_id                                        review_text  star_rating  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
1       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
2       1975  تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...            1   
3       3024                                    احلي مكان فزايد            5   
4       5483  الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...            4   

                  date              business_name      business_category  \
0  2026-03-08 00:00:00                       Noon              ecommerce   
1        قبل يومين (2)      ممشي مصر Mawlana Cafe                  كافيه   
2              قبل شهر          بيت لحم Beet Lahm                   مطعم   
3              قبل شهر  ذا بلكون كافيه الشيخ زايد  مطعم مأكولات ومشروبات   
4              قبل سنة        The Best Restaurant                   مطعم   

      platform                                 aspects  \
0   

In [44]:
print(df.columns)

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='str')


In [45]:
# Convert JSON strings into real objects
import json

df["aspects"] = df["aspects"].apply(json.loads)
df["aspect_sentiments"] = df["aspect_sentiments"].apply(json.loads)

In [46]:
# Convert ABSA into ML format
rows = []

for _, row in df.iterrows():
    review = row["review_text"]
    sentiments = row["aspect_sentiments"]

    for aspect, sentiment in sentiments.items():
        rows.append({
            "text": review,
            "aspect": aspect,
            "label": sentiment
        })

train_df = pd.DataFrame(rows)

print(train_df.head())

                                                text          aspect     label
0                لا يوجد الدفع بالبطاقه عند الاستلام  app_experience  negative
1                لا يوجد الدفع بالبطاقه عند الاستلام        delivery  negative
2  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...     cleanliness  positive
3  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...        ambiance  positive
4  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...         service  positive


In [47]:
# Convert labels to numbers
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_df["label_id"] = train_df["label"].map(label_map)

In [48]:
import re

def clean_text(text):
    # 1. حذف newline (\n)
    text = re.sub(r'\n+', ' ', text)

    # 2. حذف التواريخ
    text = re.sub(r'\d{4}-\d{2}-\d{2}', '', text)
    text = re.sub(r'\d{2}/\d{2}/\d{4}', '', text)

    # 3. underscore → space
    text = text.replace("_", " ")

    # 4. حذف الرموز
    text = re.sub(r'[@#$%^&*+!~|?><\-]', '', text)

    # 5. حذف الـ ...
    text = re.sub(r'\.{2,}', '', text)

    # 6. حذف أي punctuation زيادة
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)

    # 7. توحيد الحروف
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)

    # 8. إزالة التكرار
    text = re.sub(r'(.)\1+', r'\1\1', text)

    # 9. تنظيف المسافات
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [49]:
print(train_df.tail())

                                                   text       aspect  \
3328  The worst experience at the HUB. Place is dirt...  cleanliness   
3329  The worst experience at the HUB. Place is dirt...         food   
3330  الكوفي في فندق الهيلتون رمسيس...\n\nتم اضافه س...      service   
3331  الشغل كتير حلو ونظيف والاستقبال جميل جدا وهي ك...  cleanliness   
3332  الشغل كتير حلو ونظيف والاستقبال جميل جدا وهي ك...      service   

         label  label_id  
3328  negative         0  
3329  negative         0  
3330  negative         0  
3331  positive         2  
3332  positive         2  


In [50]:
train_df["input"] = train_df["text"] + " " + train_df["aspect"]
print(train_df[["input"]].head())

                                               input
0  لا يوجد الدفع بالبطاقه عند الاستلام app_experi...
1       لا يوجد الدفع بالبطاقه عند الاستلام delivery
2  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...
3  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...
4  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...


In [51]:
# Combine text + aspect
train_df["input"] = train_df["text"] + " " + train_df["aspect"]


In [52]:
print(train_df[["input"]].head())


                                               input
0  لا يوجد الدفع بالبطاقه عند الاستلام app_experi...
1       لا يوجد الدفع بالبطاقه عند الاستلام delivery
2  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...
3  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...
4  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...


In [53]:
import sys
print(sys.executable)

c:\Users\11\AppData\Local\Programs\Python\Python313\python.exe


In [54]:
!pip install transformers datasets torch

In [55]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

print("Model loaded successfully 🚀")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully 🚀


In [56]:
encodings = tokenizer(
    list(train_df["input"]),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([3333, 128])


In [57]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

print(f"train: {len(train_data)}")
print(f"val:   {len(val_data)}")

train: 2666
val:   667
